[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/typer-certified/notebooks/day-02-options-types-callbacks-and-testing.ipynb#scrollTo=a2b3c4d5)

---
# Day 2 · Options, Types, Callbacks & Testing
**certified-journeys / typer-certified** &nbsp;|&nbsp; Intermediate

> **Goal for today:** Master `typer.Option()`, implement a `--version` callback, and write a complete CliRunner test suite — all in 30 minutes.


In [ ]:
%pip install -q 'typer[all]'


---
## Step 1 · typer.Option() and built-in types

`typer.Option()` gives explicit control over every aspect of an optional parameter:

| Parameter | Effect |
|---|---|
| `envvar=` | Read from env var if flag not supplied |
| `prompt=True` | Interactive prompt when value missing |
| `hide_input=True` | Mask input (passwords) |
| `show_default=True` | Display default in `--help` |
| `min=`, `max=` on int | Range validation — no manual `if` needed |
| `click_type=click.Choice([...])` | Enum validation; choices shown in `--help` |


In [ ]:
import typer, click
from pathlib import Path
from typing import Optional
from typer.testing import CliRunner

runner = CliRunner()
app = typer.Typer()

@app.command()
def deploy(
    service: str = typer.Argument(..., help='Service name'),
    env: str     = typer.Option("staging", click_type=click.Choice(["staging","production","dev"]),
                                envvar="DEPLOY_ENV", help="Target environment"),
    replicas: int = typer.Option(1, min=1, max=10, help='Replica count'),
    dry_run: bool = typer.Option(False, "--dry-run", help="Preview without applying"),
):
    """Deploy SERVICE to ENV."""
    prefix = "[DRY RUN] " if dry_run else ""
    typer.echo(f"{prefix}Deploying {service} → {env} × {replicas}")

# Happy path
r = runner.invoke(app, ["api", "--env", "production", "--replicas", "3"])
print(r.output)

# Choice validation rejects invalid env
r2 = runner.invoke(app, ['api', '--env', 'invalid'])
print("invalid env exit code:", r2.exit_code)  # non-zero

# Range validation rejects 0
r3 = runner.invoke(app, ["api", "--replicas", "0"])
print("replicas=0 exit code:", r3.exit_code)   # non-zero


**What just happened?**
- **`envvar="DEPLOY_ENV"`** — if `--env` isn't supplied, Typer checks the env var before using the default; perfect for CI/CD
- `click.Choice([...])` rejects `"invalid"` automatically — no `if env not in [...]` guard needed
- `min=1, max=10` on an `int` Option → range validated before the function body even runs
- `--dry-run` as a bool Option gives you a `--no-dry-run` flag for free


---
## Step 2 · --version callback and testing

`is_eager=True` processes the option **before** Typer validates required arguments — essential for `--version` to work even when required args are absent.

Then we test everything with `CliRunner`: happy path, feature flags, and validation failures.


In [ ]:
APP_VERSION = '1.0.0'

def version_callback(value: bool):
    if value:
        typer.echo(f"deploy-tool {APP_VERSION}")
        raise typer.Exit()

app2 = typer.Typer()

@app2.command()
def run(
    service: str,
    version: bool = typer.Option(
        False, "--version", "-V",
        callback=version_callback,
        is_eager=True,
        help="Show version and exit",
    ),
):
    """Run SERVICE."""
    typer.echo(f"Running: {service}")

# --- test suite ---
def test_version():
    r = runner.invoke(app2, ["--version"])
    assert r.exit_code == 0
    assert APP_VERSION in r.output

def test_run_success():
    r = runner.invoke(app2, ["api-svc"])
    assert r.exit_code == 0
    assert "api-svc" in r.output

def test_missing_arg():
    r = runner.invoke(app2, [])
    assert r.exit_code != 0  # 'service' is required

test_version()
test_run_success()
test_missing_arg()
print("All tests passed ✓")


**What just happened?**
- **`is_eager=True`** — processes `--version` before validating required `service`; without it `--version` alone would fail
- `raise typer.Exit()` inside a callback exits cleanly with code 0
- `test_missing_arg()` asserts `exit_code != 0` — verifying Typer's own required-arg enforcement is a regression safety net
- Three tests, one `CliRunner` — covers version, success, and validation failure in under 15 lines


---
## Step 3 · Rich progress output

Typer ships with Rich. Use it for progress bars and styled output without extra dependencies.

- `Progress` context manager — auto-removes the bar on exit, even on exception
- `SpinnerColumn`, `BarColumn`, `TimeElapsedColumn` are composable
- `console.print('[bold green]✓[/bold green] Done')` — markup in any colour-capable terminal


In [ ]:
import time
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn
from rich.console import Console

rich_console = Console()
progress_app = typer.Typer()

@progress_app.command()
def process(count: int = typer.Option(4, help='Items to process')):
    """Process COUNT items with a progress bar."""
    with Progress(SpinnerColumn(), TextColumn('{task.description}'), BarColumn()) as p:
        task = p.add_task('Processing...', total=count)
        for i in range(count):
            time.sleep(0.02)
            p.update(task, advance=1, description=f'Item {i+1}/{count}')
    rich_console.print('[bold green]✓[/bold green] All done')

r = runner.invoke(progress_app, ['--count', '3'])
print('exit code:', r.exit_code)  # 0 — progress bar runs fine via CliRunner


**What just happened?**
- `Progress(...)` context manager — the bar is removed on exit even if an exception is raised
- `p.update(task, advance=1)` increments the bar; `description=` updates the label live
- CliRunner captures rich output in tests — `r.exit_code == 0` confirms the progress block ran cleanly


---
## Step 4 · Combine all Day 2 patterns

A realistic `deploy` command using envvar, Choice, int range, version callback, and a full test suite together.


In [ ]:
VALID_ENVS = ['staging', 'production', 'dev']
DEPLOY_VERSION = '2.0.0'

def deploy_version_cb(value: bool):
    if value: typer.echo(f'deploy-tool {DEPLOY_VERSION}'); raise typer.Exit()

full_app = typer.Typer()

@full_app.command()
def deploy(
    service:  str  = typer.Argument(...),
    env:      str  = typer.Option('staging', click_type=__import__('click').Choice(VALID_ENVS), envvar='DEPLOY_ENV'),
    replicas: int  = typer.Option(1, min=1, max=10),
    dry_run:  bool = typer.Option(False, '--dry-run'),
    version:  bool = typer.Option(False, '--version', '-V', callback=deploy_version_cb, is_eager=True),
):
    """Deploy SERVICE to ENV."""
    prefix = '[DRY RUN] ' if dry_run else ''
    typer.echo(f'{prefix}Deploying {service} → {env} × {replicas}')

# tests
def test_deploy_default():      r=runner.invoke(full_app,['svc']);        assert r.exit_code==0 and 'staging' in r.output
def test_deploy_prod():         r=runner.invoke(full_app,['svc','--env','production']); assert 'production' in r.output
def test_deploy_dry():          r=runner.invoke(full_app,['svc','--dry-run']); assert '[DRY RUN]' in r.output
def test_deploy_version():      r=runner.invoke(full_app,['--version']);   assert DEPLOY_VERSION in r.output
def test_deploy_bad_env():      r=runner.invoke(full_app,['svc','--env','prod']); assert r.exit_code!=0
def test_deploy_bad_replicas(): r=runner.invoke(full_app,['svc','--replicas','0']); assert r.exit_code!=0

for t in [test_deploy_default,test_deploy_prod,test_deploy_dry,
           test_deploy_version,test_deploy_bad_env,test_deploy_bad_replicas]:
    try: t(); print(f'  ✓ {t.__name__}')
    except AssertionError as e: print(f'  ✗ {t.__name__}: {e}')


**What just happened?**
- All Day 2 patterns in one command: `envvar`, `Choice`, range, `--dry-run`, and `--version` with `is_eager`
- Six tests cover every code path — including two validation-failure assertions (`exit_code != 0`)
- This is the test structure to copy for any production Typer command


---
## Step 5 · Path type and prompt option

Two more useful Option patterns: `Path` for file arguments with existence checking, and `prompt=True` for interactive input.


In [ ]:
from pathlib import Path

path_app = typer.Typer()

@path_app.command()
def read_file(
    filepath: Path = typer.Argument(..., help='File to read'),
    encoding: str  = typer.Option('utf-8', help='File encoding'),
):
    """Read and echo FILEPATH contents."""
    if not filepath.exists():
        typer.echo(f'Not found: {filepath}', err=True)
        raise typer.Exit(1)
    typer.echo(filepath.read_text(encoding=encoding))

@path_app.command()
def login(
    username: str = typer.Option(..., prompt=True,  help='Username'),
    password: str = typer.Option(..., prompt=True, hide_input=True, help='Password'),
):
    """Login with USERNAME and PASSWORD."""
    typer.echo(f'Logged in as: {username}')

# test login by passing flags (avoids interactive prompt in tests)
r = runner.invoke(path_app, ['login', '--username', 'hari', '--password', 'secret'])
print(r.output)

# test missing file → exit 1
r2 = runner.invoke(path_app, ['read-file', '/nonexistent.txt'])
print('missing file exit:', r2.exit_code)


**What just happened?**
- `Path` argument — Typer passes a `pathlib.Path` object directly; `.exists()` and `.read_text()` work immediately
- `prompt=True, hide_input=True` → interactive masked password input in production; pass `--password` in tests to skip the prompt
- `Option(..., prompt=True)` — the `...` means the option is **required** but collected via prompt if not passed as a flag


In [ ]:
# Mini test suite for the path_app above
def test_login_success():
    r = runner.invoke(path_app, ['login', '--username', 'hari', '--password', 'secret'])
    assert r.exit_code == 0 and 'hari' in r.output

def test_read_missing():
    r = runner.invoke(path_app, ['read-file', '/no/such/file.txt'])
    assert r.exit_code == 1

def test_read_real_file(tmp_path=None):
    import tempfile, os
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
        f.write('hello typer\n')
        fname = f.name
    r = runner.invoke(path_app, ['read-file', fname])
    os.unlink(fname)
    assert r.exit_code == 0 and 'hello' in r.output

test_login_success(); print('✓ test_login_success')
test_read_missing();  print('✓ test_read_missing')
test_read_real_file();print('✓ test_read_real_file')


In [ ]:
# Challenge: extend app2's deploy command with a --password option:
#   - prompt=True, hide_input=True
#   - Write a test that passes '--password secret' directly (avoids interactive prompt)
#   - Assert exit_code == 0 and 'Deploying' in output

# Your solution here


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `envvar=` on Option | CI/CD friendly — env var fills in flag when not supplied |
| `click.Choice([...])` | Enum validation, shown in `--help`, zero extra code |
| `min=`/`max=` on int | Range validated before your function runs |
| `is_eager=True` | Process option before required args — needed for `--version` |
| `CliRunner.invoke()` | Test happy path + validation failures by checking `.exit_code` |

> **Tip:** Always test your CLIs with CliRunner — it catches argument-parsing bugs you'd never find manually. A single `invoke()` call covers the full execution path.

---
## What's next

**Day 3** → Build a complete pip-installable CLI tool: multi-command app, pyproject.toml entry point, and a full pytest test suite.

Mark Day 2 complete in your [tracker](../index.html).
